# 06 Reasoning Token Budget: gpt-4o vs gpt-5.1

This notebook demonstrates that the same `max_completion_tokens` budget can behave differently across the two deployments in this demo.

The key behavior to watch for is this:
- `gpt-4o` often starts emitting visible text under smaller budgets.
- `gpt-5.1` can spend much more of the completion budget on reasoning before any visible answer appears.
- When that happens, you may see a `200` response with `finish_reason = "length"`, non-zero `reasoning_tokens`, and an empty visible response.

## Load Environment

This notebook uses the same direct Azure OpenAI deployments configured by `01_deploy_infra.ipynb`.

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

env_path = Path('../env/.env')
if not env_path.exists():
    raise RuntimeError('env/.env not found. Run 01_deploy_infra.ipynb first.')

load_dotenv(env_path)

AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '').rstrip('/')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY', '')
PRIMARY_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'chat4o')
SECONDARY_DEPLOYMENT = os.getenv('AZURE_OPENAI_SECONDARY_DEPLOYMENT', 'chat51')
API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')

if not AZURE_OPENAI_API_KEY:
    raise RuntimeError('AZURE_OPENAI_API_KEY is missing. Run 01_deploy_infra.ipynb first.')

print(f'Endpoint:             {AZURE_OPENAI_ENDPOINT}')
print(f'Primary deployment:   {PRIMARY_DEPLOYMENT}  (gpt-4o)')
print(f'Secondary deployment: {SECONDARY_DEPLOYMENT}  (gpt-5.1)')
print(f'API version:          {API_VERSION}')

## Define Probe Prompts and Budgets

These prompts are intentionally chosen to make `gpt-5.1` do more internal reasoning before it writes visible output.

The signature we are trying to catch is:
- `status_code = 200`
- `finish_reason = length`
- `reasoning_tokens` consumes most or all completion tokens
- `visible_chars = 0`

In [ ]:
PROBE_SCENARIOS = [
    {
        'id': 'code_review_heavy',
        'label': 'Security review with explanation',
        'system': (
            'You are a security-conscious code reviewer. Identify vulnerabilities and explain '
            'each one clearly with its OWASP category.'
        ),
        'user': (
            'Review this Python snippet:\n\n'
            '```python\n'
            'import sqlite3\n'
            'def get_user(username):\n'
            '    conn = sqlite3.connect("users.db")\n'
            '    cursor = conn.cursor()\n'
            '    cursor.execute(f"SELECT * FROM users WHERE name = \'{username}\'")\n'
            '    return cursor.fetchall()\n'
            '```'
        ),
    },
    {
        'id': 'shakespeare_rest',
        'label': 'Style-heavy instruction following',
        'system': (
            'You always respond in the style of a Shakespearean play, using verse where natural. '
            'Stay fully in character; never break the fourth wall.'
        ),
        'user': 'Explain what a REST API is.',
    },
]

BUDGETS = [120, 250, 400, 800, 2000]

print('Scenarios:')
for scenario in PROBE_SCENARIOS:
    print(f"- {scenario['id']}: {scenario['label']}")

print('Budgets:', BUDGETS)

## Run the Probe

This cell sends the same prompt and token budget to both deployments and captures:
- `finish_reason`
- `completion_tokens`
- `reasoning_tokens`
- visible response length
- a short preview of visible text

In [ ]:
import requests

def extract_visible_text(choice):
    message = choice.get('message') or {}
    content = message.get('content', '')

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(item.get('text', ''))
            else:
                parts.append(str(item))
        content = ''.join(parts)

    return (content or '').strip()


def call_model(deployment, model_label, scenario, token_budget):
    url = (
        f'{AZURE_OPENAI_ENDPOINT}/openai/deployments/{deployment}'
        f'/chat/completions?api-version={API_VERSION}'
    )
    headers = {
        'api-key': AZURE_OPENAI_API_KEY,
        'Content-Type': 'application/json',
    }
    payload = {
        'messages': [
            {'role': 'system', 'content': scenario['system']},
            {'role': 'user', 'content': scenario['user']},
        ],
        'max_completion_tokens': token_budget,
        'temperature': 0.7,
    }

    response = requests.post(url, headers=headers, json=payload, timeout=90)

    row = {
        'scenario_id': scenario['id'],
        'scenario_label': scenario['label'],
        'model_label': model_label,
        'deployment': deployment,
        'token_budget': token_budget,
        'status_code': response.status_code,
        'finish_reason': '',
        'completion_tokens': None,
        'reasoning_tokens': 0,
        'visible_chars': 0,
        'empty_visible_response': False,
        'response_preview': '',
        'model': '',
        'error_text': '',
    }

    if response.status_code != 200:
        row['error_text'] = response.text[:400]
        return row

    body = response.json()
    choice = (body.get('choices') or [{}])[0]
    usage = body.get('usage') or {}
    completion_details = usage.get('completion_tokens_details') or {}
    visible_text = extract_visible_text(choice)

    row['finish_reason'] = choice.get('finish_reason', '') or ''
    row['completion_tokens'] = usage.get('completion_tokens')
    row['reasoning_tokens'] = completion_details.get('reasoning_tokens', 0) or 0
    row['visible_chars'] = len(visible_text)
    row['empty_visible_response'] = len(visible_text) == 0
    row['response_preview'] = visible_text.replace('\n', ' ')[:180]
    row['model'] = body.get('model', '')
    return row


probe_results = []

for scenario in PROBE_SCENARIOS:
    print(f"\n=== {scenario['label']} ===")
    for token_budget in BUDGETS:
        for deployment, model_label in [
            (PRIMARY_DEPLOYMENT, 'gpt-4o'),
            (SECONDARY_DEPLOYMENT, 'gpt-5.1'),
        ]:
            row = call_model(deployment, model_label, scenario, token_budget)
            probe_results.append(row)

            if row['status_code'] == 200:
                print(
                    f"{model_label:7} budget={token_budget:4} finish={row['finish_reason'] or '[none]':7} "
                    f"visible_chars={row['visible_chars']:4} completion_tokens={str(row['completion_tokens']):4} "
                    f"reasoning_tokens={row['reasoning_tokens']:4}"
                )
            else:
                print(f"{model_label:7} budget={token_budget:4} status={row['status_code']} error={row['error_text'][:120]!r}")

print(f'\nCollected {len(probe_results)} probe rows.')

## Compare the Results

The table below makes the empty-visible-response pattern obvious.

If `gpt-5.1` shows `finish_reason = length`, `reasoning_tokens ~= completion_tokens`, and `visible_chars = 0`, the completion budget was spent before the visible answer began.

In [ ]:
from IPython.display import Markdown, display

def row_note(row):
    if row['status_code'] != 200:
        return f"HTTP {row['status_code']}"
    if row['empty_visible_response'] and row['finish_reason'] == 'length' and row['reasoning_tokens']:
        return 'Budget exhausted on reasoning before visible output'
    if row['empty_visible_response']:
        return 'No visible output returned'
    if row['finish_reason'] == 'length':
        return 'Visible output returned, but response hit token limit'
    return 'Visible output returned'


lines = [
    '### Probe Results',
    '',
    '| Scenario | Model | Budget | Finish | Visible chars | Completion tokens | Reasoning tokens | Notes |',
    '|---|---|---:|---|---:|---:|---:|---|'
]

for row in probe_results:
    scenario = row['scenario_label'].replace('|', '\\|')
    model_label = row['model_label']
    finish = (row['finish_reason'] or '[none]').replace('|', '\\|')
    note = row_note(row).replace('|', '\\|')
    lines.append(
        f"| {scenario} | {model_label} | {row['token_budget']} | {finish} | {row['visible_chars']} | "
        f"{row['completion_tokens'] if row['completion_tokens'] is not None else ''} | {row['reasoning_tokens']} | {note} |"
    )

display(Markdown('\n'.join(lines)))

## Summarize the Threshold

This summary highlights the smallest budget where each model first produces visible output for each scenario.

In [ ]:
from collections import defaultdict

first_visible = {}
for row in probe_results:
    key = (row['scenario_id'], row['model_label'])
    if row['status_code'] == 200 and row['visible_chars'] > 0 and key not in first_visible:
        first_visible[key] = row

lines = [
    '### First Visible Output by Model',
    '',
    '| Scenario | Model | First budget with visible output | Evidence |',
    '|---|---|---:|---|'
]

for scenario in PROBE_SCENARIOS:
    for model_label in ['gpt-4o', 'gpt-5.1']:
        key = (scenario['id'], model_label)
        if key in first_visible:
            row = first_visible[key]
            evidence = f"finish={row['finish_reason'] or '[none]'}, visible_chars={row['visible_chars']}, reasoning_tokens={row['reasoning_tokens']}"
            budget_text = str(row['token_budget'])
        else:
            evidence = 'No visible output within tested budgets'
            budget_text = 'n/a'

        lines.append(
            f"| {scenario['label']} | {model_label} | {budget_text} | {evidence.replace('|', '\\|')} |"
        )

display(Markdown('\n'.join(lines)))

## Save Probe Output

Writes the raw probe rows to `outputs/reasoning-token-budget-results.jsonl` so you can diff runs later.

In [ ]:
output_path = Path('../outputs/reasoning-token-budget-results.jsonl')
output_path.parent.mkdir(parents=True, exist_ok=True)

lines = [json.dumps(row, ensure_ascii=False) for row in probe_results]
output_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(f'Saved {len(probe_results)} rows to {output_path}')

## Summary

Use this notebook when you want to show that `max_completion_tokens` is not equivalent to visible-output budget across models.

In this demo, the important pattern is:
- `gpt-4o` often starts writing sooner.
- `gpt-5.1` may consume more completion budget on reasoning first.
- An empty response does not necessarily mean the request failed; it can mean the model hit the token ceiling before visible output began.